# ABL Bio's Current Patent Portfolio (as of Feb 26, 2026)
Look into patent's ABL Bio owns using Lens.org
- Use search term `applicant.name:"ABL Bio" OR applicant.name:"ABLBIO"` or `owner.name:"ABL Bio" OR owner.name:"ABLBIO"`
- For applicant -> Results will show 466 Patent Records (Expand by Simple Family), but after selecting simple families, there are 37 results
- For owner -> Results will show 437 Patent Records (Expand by Simple Family), but 24 Simple families
- Download results for the 37 and 24 Families as `applicant-abl-bio.csv` and `owner-abl-bio.csv`

<u>Note:</u>
- Applicant will tell: What ABL Bio originated
- Owner will tell: What ABL Bio currently owns/controls

This suggests that
- ABL Bio has original 37 simple families ("inventions")
- But is currently in control of 24

In [97]:
import pandas as pd

app_abl = pd.read_csv("../data/raw/applicant-abl-bio.csv")
own_abl = pd.read_csv("../data/raw/owner-abl-bio.csv")

print(app_abl.shape)
print(own_abl.shape)

(37, 37)
(24, 37)


In [98]:
app_fams = set(app_abl['Simple Family Members'])
own_fams = set(own_abl['Simple Family Members'])

overlap = app_fams & own_fams
app_only = app_fams - own_fams
own_only = own_fams - app_fams

len(overlap), len(app_only), len(own_only)

(24, 13, 0)

So ABL Bio:
- Originated 37 simple families
- Currently owns 24
- Owns zero families they did not originate

Suggests -> No acquisition-based portfolio growth

# Key Inventors

1) Inventors of current portfolio (owner dataset)
- Looking into enforceable IP

2) Inventors in originated portfolio (applicant dataset)
- Historically, who were the inventors that drove innovation/development?

3) Compare the overlap
- Did key inventors leave?
- Are certain inventors tied to abandoned assets?


In [99]:
own_abl['Inventors'].head()

0    LEE DONG HEON;;MOON KYUNG DUK;;CHOI YU BIN;;KA...
1    KIM EUN A;;PARK SANG KYUNG;;MOON KYUNG DUK;;LE...
2    KIM YOUNG MIN;;KO MIN JI;;KIM JAE YONG;;KIM JU...
3    PARK EUNYOUNG;;LEE YANGSOON;;CHUNG HYEJIN;;JUN...
4    PARK KYUNGJIN;;CHUNG HYEJIN;;PARK KYEONGSU;;LE...
Name: Inventors, dtype: object

In [100]:
own_inv = (
    own_abl
    .assign(Inventors=own_abl['Inventors'].str.split(';;'))
    .explode('Inventors')
)

own_inv['Inventors'] = own_inv['Inventors'].str.strip()

owner_counts = own_inv['Inventors'].value_counts()

owner_counts.head(10)

Inventors
SUNG BYUNGJE     11
JUNG JINWON      11
LEE YANGSOON     10
KIM YEUNJU       10
PAK YOUNGDON     10
SUNG EUNSIL       9
SONG DAEHAE       9
AHN JINHYUNG      8
KIM JUHEE         8
PARK KYUNGJIN     8
Name: count, dtype: int64

In [101]:
app_inv = (
    app_abl
    .assign(Inventors=app_abl['Inventors'].str.split(';;'))
    .explode('Inventors')
)

app_inv['Inventors'] = app_inv['Inventors'].str.strip()

app_counts = app_inv['Inventors'].value_counts()

app_counts.head(10)

Inventors
LEE YANGSOON     14
SUNG BYUNGJE     13
JUNG JINWON      13
PAK YOUNGDON     12
KIM JUHEE        12
PARK KYUNGJIN    12
AHN JINHYUNG     11
CHUNG HYEJIN     10
LEE BORA         10
KIM YEUNJU       10
Name: count, dtype: int64

Results suggest:
- Core inventors are stable (top 4 is the same names)
- Some inventors appear strong in applicant but weaker in owner. For example, Kyungjin and Juhee appears as 12 applicants -> 8 owners (Could just be some of their earlier families were pruned or some projects were dropped)
- Have about ~5–7 core inventors, that each seem to be appearing in ~40-50% of ABL Bio's owned simple families

Due the time and purpose of this case study, I will be looking into patents of just the top 5 inventors for each list
- SUNG BYUNGJE
- JUNG JINWON
- LEE YANGSOON
- KIM YEUNJU
- PAK YOUNGDON
- KIM JUHEE

So inputting into Lens.org:

(inventor.name:"SUNG BYUNGJE" OR  
inventor.name:"JUNG JINWON" OR  
inventor.name:"PAK YOUNGDON" OR  
inventor.name:"KIM YEUNJU" OR  
inventor.name:"LEE YANGSOON" OR  
inventor.name:"KIM JUHEE")  
AND NOT (applicant.name:"ABL Bio"  
OR applicant.name:"ABLBIO"  
OR owner.name:"ABL Bio"  
OR owner.name:"ABLBIO")

But also to limit mis-identified inventors (due to same name), will try narrow it down to:  
AND (class_cpc.symbol:C07K16/*  
OR class_cpc.symbol:C07K2317/*  
OR class_cpc.symbol:A61K39/*  
OR class_cpc.symbol:A61K47/*)  

- C07K16/* → antibodies
- C07K2317/* → conjugates involving peptides/antibodies
- A61K39/* → medicinal antibodies
- A61K47/* → formulations & carriers (relevant for ADC linkers)

This will result in 10 simple families
- So 10 ADC-class patents, that involve ABL Bio’s core inventors but are not currently associated with ABL Bio as either applicant or owner
- Saved this as `inventor-no-abl.csv`

In [102]:
inv_ext = pd.read_csv("../data/raw/inventor-no-abl.csv")

print(inv_ext['Legal Status'].value_counts(),"\n")
print(inv_ext['Applicants'].value_counts().head(10))


Legal Status
PENDING    9
ACTIVE     1
Name: count, dtype: int64 

Applicants
YUHAN CORP                                                                                                                                                                                   3
TSD LIFE SCIENCES CO LTD                                                                                                                                                                     1
I MAB BIOPHARMA HANGZHOU CO LTD                                                                                                                                                              1
LEGOCHEM BIOSCIENCES INC                                                                                                                                                                     1
I MAB BIOPHARMA US LTD                                                                                                                                        

# Current Portfolio - Legal Status

In [104]:
own_abl['Legal Status'].value_counts() #Recall there are 24 families ABL Bio owns

Legal Status
ACTIVE          13
PENDING         10
DISCONTINUED     1
Name: count, dtype: int64

In [105]:
#Legal Status Breakdown:
print(own_abl['Legal Status'].value_counts())
print("\n")
status_counts = own_abl['Legal Status'].value_counts(normalize=True) * 100
print(status_counts)

Legal Status
ACTIVE          13
PENDING         10
DISCONTINUED     1
Name: count, dtype: int64


Legal Status
ACTIVE          54.166667
PENDING         41.666667
DISCONTINUED     4.166667
Name: proportion, dtype: float64


In [108]:
#Jurisdiction Coverage in Major Markets
own_abl['has_us'] = own_abl['Extended Family Member Jurisdictions'].str.contains('US', na=False) #USA
own_abl['has_ep'] = own_abl['Extended Family Member Jurisdictions'].str.contains('EP', na=False) #Europe
own_abl['has_cn'] = own_abl['Extended Family Member Jurisdictions'].str.contains('CN', na=False) #China
own_abl['has_jp'] = own_abl['Extended Family Member Jurisdictions'].str.contains('JP', na=False) #Japan
own_abl['has_wo'] = own_abl['Extended Family Member Jurisdictions'].str.contains('WO', na=False) #PCT

coverage = {
    'US': own_abl['has_us'].sum(),
    'EP': own_abl['has_ep'].sum(),
    'CN': own_abl['has_cn'].sum(),
    'JP': own_abl['has_jp'].sum(),
    'WO': own_abl['has_wo'].sum()

}

print(coverage)

{'US': 24, 'EP': 24, 'CN': 24, 'JP': 24, 'WO': 24}


In [109]:
# Overall Filing Distribution

#split jurisdictions into lists (they are seperated by ;;)
own_abl['jurisdiction_list'] = own_abl['Extended Family Member Jurisdictions'].str.split(';;')

#explode into long format
jur_long = own_abl.explode('jurisdiction_list')

#clean whitespace
jur_long['jurisdiction_list'] = jur_long['jurisdiction_list'].str.strip()

#count frequency
jur_counts = jur_long['jurisdiction_list'].value_counts()

print(jur_counts)

jurisdiction_list
CN    24
EP    24
KR    24
WO    24
US    24
JP    24
CA    19
AU    18
BR    15
MX    12
IL     7
EA     5
ZA     5
CO     4
PE     3
CL     3
PH     3
TW     2
RU     2
SG     2
MY     2
NZ     2
ES     1
TR     1
AR     1
PL     1
Name: count, dtype: int64


# Portfolio Structure 

Since there is 24 families, will manually annotate what each patent is for

In [121]:
for i, row in own_abl.iterrows():
    print(f"\n--- Family {i+1} ---")
    print("Title:")
    print(row['Title'])
    print("\nCPC:")
    print(row['CPC Classifications'])


--- Family 1 ---
Title:
Novel dual-targeted protein specifically binding to DLL4 and VEGF, and use thereof

CPC:
C07K16/22;;C07K16/28;;C07K2317/31;;C07K2317/622;;C07K2317/64;;A61K2039/505;;C07K16/468;;C07K2317/34;;C07K2317/76;;C07K2317/92;;A61P1/02;;A61P1/04;;A61P1/16;;A61P1/18;;A61P11/00;;A61P13/08;;A61P13/10;;A61P13/12;;A61P15/00;;A61P17/00;;A61P19/00;;A61P25/00;;A61P35/00;;A61P35/02;;A61P5/00;;C07K16/2803;;C07K16/286;;C07K2317/622;;A61K39/00;;C07K1/00;;C07K16/00;;C07K16/46;;C07K16/468;;C07K16/22;;C07K16/28;;C07K2317/31;;C07K2317/622;;C07K2317/64;;G01N33/5759;;G01N33/57585;;A61K2039/505;;C07K16/468;;C07K2317/34;;C07K2317/76;;C07K2317/92;;G01N2333/475;;G01N2333/705

--- Family 2 ---
Title:
NOVEL MONOCLONAL ANTIBODY BINDING SPECIFICALLY TO DLL4 AND USE THEREOF

CPC:
C07K16/22;;C07K16/28;;G01N33/577;;A61K2039/505;;C07K2317/92;;C07K2317/76;;G01N2333/4703;;G01N33/575;;C07K16/22;;C07K16/28;;C07K2317/76;;C07K2317/92;;G01N2333/4703;;A61P1/04;;A61P11/00;;A61P11/06;;A61P17/00;;A61P17/06;;A61P

#### **The distribution looks like:**
<u>1) Purely Antibody Composition Patents:</u> (n=9)
- Families 2, 4, 5, 7, 9, 12, 13, 15 and 19
- Dominant CPC of C07K16/* (Immunoglobulins, e.g., monoclonal or polyclonal antibodies)
- For DLL4, 4-1BB, BCMA, alpha-synuclein, B7-H3, ROR1, CLL-1 

<u>2) Bi-specific Antibody Composition Patents:</u> (n=10)
- Families 1, 8, 11, 14, 17, 18, 20, 21, 22 and 24
- For DLL4/VEGF, Alpha-SYN/IGF1R, PD-L1/LAG3, B7-H4/4-1BB, BCMA/4-1BB, HER2/4-1BB, EGFR/4-1BB, PD-L1/B7-H3, Claudin 18.2/4-1BB

<u>3) ADC Composition Patents (Conjugate):</u> (n=3)
- Families 3, 16, and 23
- Abstracts are respectively:  
   - "The present invention relates to an antibody-drug conjugate comprising a drug conjugated to an antibody, a preparation method thereof and the use thereof."  
   - "The present invention relates to a novel compound comprising a galactose trigger moiety and cyclopropabenzindole (CBI), and an antibody-drug conjugate prepared by using same."  
   - "The present invention relates to new antibody-drug conjugates (ADCs) targeting ROR1, active metabolites of such ADCs, methods for preparation of such ADCs, uses for such ADCs in treatment and/or prevention of illnesses, and uses for such ADCs in production of drugs for treatment and/or prevention of diseases, more specifically diseases associated with over-expression of ROR1, for example cancer. More specifically, the present invention relates to an antibody-drug conjugate comprising an antibody that binds to ROR1 or an antigen-binding fragment thereof, and a pharmaceutical composition comprising the same."
- Family 3 is a **Conjugated Linker** patent: Conjugating any drug to the N-terminal α-amine of any antibody via a linker with a reactive aldehyde (breadth in antibody, payload, target)
- Family 16 is **Payload-centric** patent: Cyclopropabenzindole (CBI) cytotocix payload linked to a galactose trigger moiety (breadth in antibody, target)
- Family 23 is a **Full ADC** patent: Sequence-defined anti-ROR1 antibodies conjugated to a cytotoxic payload (broad list of active agent with focus on PBD dimers) with broad examples of linkers. Also covers treatment of ROR1-overexpressing cancers, a long medical indication list (CLL, AML, breast, NSCLC, etc.), pharmaceutical compositions and combination therapy

<u> 4) Clinical Filing:</u> (n=1)
- Family 10, with title "A method of treating a solid tumor"
- Summary is treating solid tumors using a CLDN18.2 bispecific antibody, with specified dosing ranges, schedules, routes, and patient populations (tumor types, advanced/metastatic setting)
- Suggest movement into clinical development (using in patients) and initial protection of a clinical label

<u>5) CMC Patent:</u> (n=1)
- Family 6, with title "Method for Purifying Biologically Active Peptide by Using Protein A Affinity Chromatography"
- Summary is improved purification process for Fc-containing biologics using differential Protein A binding based on VH3 domain number
    - Wild-type protein A binds to Fc region of IgG (primary) as well as VH3 family variable dowain (secondary). They determined that Protein A binding strength increases with more VH3 domains in the molecule
    - So instead of mutating Fc to artifically change affinity, use stepwise elution that will start with mild pH and then decrease -> molecules with weaker Protein A affinity elute first, followed by stronger binders at lower pH. This enables separation of highly similar antibody species


# ABL Bio's Bi-specific Antibody Platforms:
- Grabody-T
- Grabody-B
- siRNA x Grabody-B
- ADC
- Bs ADC
- Dual payload Bs ADC


# Assets that are Licensed Out